In [ ]:
import os
import gymnasium as gym
import numpy as np
import json
from collections import deque

# ĐỔI THƯ VIỆN SANG SB3-CONTRIB
from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.evaluation import evaluate_policy
from sb3_contrib.common.wrappers import ActionMasker
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback, CallbackList, BaseCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv

from mahjong_env import MahjongEnv

class SlidingWindowStatsCallback(BaseCallback):
    def __init__(self, window_size=10, verbose=0):
        super(SlidingWindowStatsCallback, self).__init__(verbose)
        self.window_size = window_size
        
        # Cửa sổ trượt lưu danh sách các ván thắng của 'window_size' iteration gần nhất
        self.history_window = deque(maxlen=window_size)
        
        # Biến tạm để lưu các ván thắng trong iteration HIỆN TẠI
        self.current_iteration_wins = []
        
        # Bộ đếm bước cho từng môi trường chạy song song
        self.env_step_counts = None

    def _on_training_start(self) -> None:
        # Khởi tạo mảng đếm bước (ví dụ: mảng 8 phần tử nếu n_envs=8)
        n_envs = self.training_env.num_envs
        self.env_step_counts = np.zeros(n_envs, dtype=int)

    def _on_step(self) -> bool:
        # Cộng thêm 1 bước cho tất cả các môi trường đang chạy
        self.env_step_counts += 1
        
        # Kiểm tra xem có môi trường nào vừa kết thúc ván đấu không
        for i, done in enumerate(self.locals['dones']):
            if done:
                info = self.locals['infos'][i]
                
                # Kiểm tra cờ báo hiệu AI tới bài (thắng)
                if info.get('shanten') == -1: 
                    # Lưu số bước của ván thắng này vào list của iteration hiện tại
                    self.current_iteration_wins.append(self.env_step_counts[i])
                
                # Reset bộ đếm bước của môi trường này về 0 cho ván mới
                self.env_step_counts[i] = 0
                
        return True

    def _on_rollout_end(self) -> None:
        # Sự kiện này tự động kích hoạt khi KẾT THÚC 1 ITERATION (trước khi học)
        
        # 1. Đẩy danh sách các ván thắng của iteration này vào cửa sổ trượt
        # Nếu history_window đã đủ 10 phần tử, nó sẽ tự đẩy phần tử cũ nhất ra
        self.history_window.append(self.current_iteration_wins)
        
        # 2. Gom tất cả số bước thắng trong toàn bộ cửa sổ (10 iterations) lại
        all_wins_in_window = []
        for iteration_wins in self.history_window:
            all_wins_in_window.extend(iteration_wins)
            
        # 3. Tính trung bình và ghi log lên Tensorboard
        if len(all_wins_in_window) > 0:
            avg_steps = np.mean(all_wins_in_window)
            
            # Ghi vào Tensorboard
            self.logger.record("mahjong_stats/avg_steps_to_win_10_iters", avg_steps)
            self.logger.record("mahjong_stats/total_wins_10_iters", len(all_wins_in_window))
            
            if self.verbose > 0:
                print(f"\n[Thống kê Iteration] Có {len(self.current_iteration_wins)} ván thắng.")
                print(f"[Cửa sổ 10 Iterations] Trung bình: {avg_steps:.1f} bước/ván (Dựa trên tổng {len(all_wins_in_window)} ván).")
        else:
            if self.verbose > 0:
                print("\n[Cửa sổ 10 Iterations] Chưa có ván thắng nào được ghi nhận.")
                
        # 4. Làm sạch danh sách tạm để chuẩn bị cho Iteration tiếp theo
        self.current_iteration_wins = []

# Tạo một hàm bọc (factory function) để sinh ra môi trường đã được mask
def make_masked_env():
    def _init():
        env = MahjongEnv()
        env = Monitor(env)
        env = ActionMasker(env, mask_fn)
        return env
    return _init

# Hàm lấy mask từ môi trường
def mask_fn(env: gym.Env) -> np.ndarray:
    # Lấy action_mask từ class MahjongEnv của bạn
    return env.unwrapped._action_mask()

def main():
    # Số lượng bàn cờ chạy song song (tùy thuộc vào số nhân CPU của bạn, ví dụ: 4 hoặc 8)
    n_envs = 8

    print(f"Khởi tạo {n_envs} môi trường chạy song song...\n")
    
    # SubprocVecEnv sẽ ép Python mở nhiều Process thực sự để chạy song song
    env = SubprocVecEnv([make_masked_env() for _ in range(n_envs)])

    # Khởi tạo MaskablePPO như cũ...
    model = MaskablePPO(
        "MlpPolicy", 
        env, 
        device="cpu", 
        learning_rate=3e-4,
        # Lưu ý: n_steps lúc này là SỐ BƯỚC MỖI MÔI TRƯỜNG. 
        # Tổng batch size = n_steps * n_envs (ví dụ: 512 * 4 = 2048)
        n_steps=512, 
        batch_size=64,
        verbose=1,
        tensorboard_log="./logs_tensorboard/"
    )

    # Khởi tạo các callback
    checkpoint_callback = CheckpointCallback(
        save_freq=500_000,
        save_path="./models/checkpoints/",
        name_prefix="mahjong_model"
    )
    
    stats_callback = SlidingWindowStatsCallback(window_size=10, verbose=1)

    # Gộp chúng lại thành 1 list
    callback_list = CallbackList([checkpoint_callback, stats_callback])

    # Chuyền list này vào model.learn()
    model.learn(
        total_timesteps=2_000_000, 
        progress_bar=False,
        callback=callback_list,  # <--- Gắn vào đây
        tb_log_name="PPO_Adaptive_Penalty_Run_1"
    )

    # Tạo thư mục chứa các bản lưu an toàn
    os.makedirs("./models/checkpoints/", exist_ok=True)
    
    # Cài đặt tự động lưu sau mỗi 500,000 bước
    checkpoint_callback = CheckpointCallback(
        save_freq=500_000,
        save_path="./models/checkpoints/",
        name_prefix="mahjong_model"
    )

    # 3. Thiết lập số step
    print("=== BẮT ĐẦU HUẤN LUYỆN ===")
    model.learn(
        total_timesteps=2_000_000, 
        progress_bar=True,
        callback=checkpoint_callback,
        tb_log_name="PPO_Adaptive_Penalty_Run_1"  # <--- TÊN CỦA ĐỢT TRAIN
    )

    # 4. Lưu mô hình bản chốt cuối cùng
    os.makedirs("models", exist_ok=True)
    model.save("models/maskable_ppo_mahjong_final")

    # 5. Chạy thử và xuất file JSON Replay
    print("\n=== CHẠY THỬ VÀ XUẤT REPLAY JSON ===")
    obs, info = env.reset()
    done = False
    step_count = 0

    # Khởi tạo cấu trúc dữ liệu Replay
    replay_data = {
        "metadata": {
            "player": "AI_Agent_Maskable",
            "game_type": "1_vs_3_Dummies"
        },
        # Lưu lại bộ bài 13 lá ban đầu của AI (chuyển đổi từ mảng NumPy sang list)
        "initial_hand": obs[:34].astype(int).tolist(), 
        "timeline": []
    }

    while not done:
        step_count += 1
        
        # 1. Lấy mask hiện tại và cho AI suy nghĩ
        action_masks = info.get("action_mask")
        action, _states = model.predict(obs, action_masks=action_masks, deterministic=True)
        
        # 2. Thực thi hành động vào môi trường
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        # 3. Đóng gói dữ liệu của lượt đi này vào lịch sử (timeline)
        turn_log = {
            "turn": step_count,
            "agent_discarded": int(action),
            "dummies_discarded": info.get("dummy_discards", []),
            "agent_drew": info.get("agent_drawn", -1),
            "shanten_after": info["shanten"],
            "wall_left": info["wall_remaining"]
        }
        replay_data["timeline"].append(turn_log)
        
        # In log ra màn hình terminal
        print(f"Bước {step_count:2d} | Đánh: {action:2d} | Bốc: {info.get('agent_drawn', -1):2d} | Shanten: {info['shanten']} | {info['shanten_detail']['status']}")
        
    # 4. Kết thúc ván đấu và lưu kết quả
    if terminated:
        print("=> AI đã TỚI BÀI!")
        replay_data["result"] = "Winning"
    elif truncated:
        print("=> Hòa.")
        replay_data["result"] = "Draw"

    # 5. Viết toàn bộ dữ liệu ra file mahjong_replay.json
    with open("mahjong_replay.json", "w", encoding="utf-8") as f:
        json.dump(replay_data, f, indent=4, ensure_ascii=False)

    print("=> Toàn bộ diễn biến đã được lưu vào file 'mahjong_replay.json'!")

if __name__ == "__main__":
    main()

In [ ]:
import os
import gymnasium as gym
import numpy as np
import json
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from mahjong_env import MahjongEnv

# 1. Định nghĩa lại hàm lấy Mask (phải giống hệt lúc train)
def mask_fn(env: gym.Env) -> np.ndarray:
    return env.unwrapped._action_mask()

def run_and_export_replay(model_path, output_file="mahjong_replay.json"):
    # 2. Khởi tạo môi trường và bọc Masker
    env = MahjongEnv()
    env = ActionMasker(env, mask_fn)

    # 3. Tải mô hình đã lưu
    if not os.path.exists(model_path):
        print(f"Lỗi: Không tìm thấy file model tại {model_path}")
        return

    print(f"Đang tải model từ: {model_path}...")
    model = MaskablePPO.load(model_path, env=env)

    # 4. Bắt đầu ván đấu
    obs, info = env.reset()
    done = False
    step_count = 0

    # Cấu trúc dữ liệu Replay
    replay_data = {
        "metadata": {
            "model_used": model_path,
            "player": "AI_Agent",
            "date_exported": "2026-04-22"
        },
        "initial_hand": obs[:34].astype(int).tolist(), 
        "timeline": []
    }

    print("\n=== AI BẮT ĐẦU ĐÁNH THỬ 1 VÁN ===")
    
    while not done:
        step_count += 1
        
        # Lấy mask và để AI dự đoán (deterministic=True để AI đánh chuẩn nhất, không bốc đồng)
        action_masks = info.get("action_mask")
        action, _states = model.predict(obs, action_masks=action_masks, deterministic=True)
        
        # Thực thi hành động
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        # Lưu log lượt đi
        turn_log = {
            "turn": step_count,
            "agent_discarded": int(action),
            "dummies_discarded": info.get("dummy_discards", []),
            "agent_drew": info.get("agent_drawn", -1),
            "shanten_after": info["shanten"],
            "shanten_status": info["shanten_detail"]["status"],
            "reward_this_step": float(reward)
        }
        replay_data["timeline"].append(turn_log)
        
        # In nhanh ra màn hình để theo dõi
        print(f"Lượt {step_count:2d} | AI đánh: {action:2d} | Shanten: {info['shanten']} | Reward: {reward:.2f}")

    # 5. Kết luận ván đấu
    if terminated:
        print("\n🎉 KẾT QUẢ: AI ĐÃ TỚI BÀI (WIN)!")
        replay_data["result"] = "Winning"
    else:
        print("\n🤝 KẾT QUẢ: HÒA (DRAW).")
        replay_data["result"] = "Draw"

    # 6. Xuất ra file JSON
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(replay_data, f, indent=4, ensure_ascii=False)
    
    print(f"\n=> Đã lưu diễn biến ván đấu vào file: {output_file}")

if __name__ == "__main__":
    # Thay đổi đường dẫn tới file model của bạn ở đây
    MODEL_FILE = "models/checkpoints_1/mahjong_model_1000000_steps.zip" 
    run_and_export_replay(MODEL_FILE)

In [ ]:
import os
import gymnasium as gym
import numpy as np
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from mahjong_env import MahjongEnv

# 1. Hàm lấy mask
def mask_fn(env: gym.Env) -> np.ndarray:
    return env.unwrapped._action_mask()

def evaluate_last_5_steps(model_path, num_episodes=100):
    # 2. Khởi tạo môi trường
    env = MahjongEnv()
    env = ActionMasker(env, mask_fn)

    if not os.path.exists(model_path):
        print(f"Lỗi: Không tìm thấy file model tại {model_path}")
        return

    print(f"Đang tải model từ: {model_path}...")
    model = MaskablePPO.load(model_path, env=env)

    # Dictionary để lưu tổng số lần xuất hiện của các mức Shanten trong 5 bước cuối
    # Mình theo dõi từ -1 (Tới bài) cho đến 6 (Rác)
    total_shanten_counts = {i: 0 for i in range(-1, 9)}
    
    print(f"\n🚀 Đang mô phỏng {num_episodes} ván đấu (Vui lòng đợi vài giây)...")

    # 3. Vòng lặp mô phỏng
    for ep in range(num_episodes):
        obs, info = env.reset()
        done = False
        
        # Mảng lưu trữ trạng thái Shanten qua từng bước của ván này
        # Bắt đầu bằng Shanten lúc vừa chia bài xong
        shanten_history = [info['shanten']] 

        while not done:
            action_masks = info.get("action_mask")
            action, _states = model.predict(obs, action_masks=action_masks, deterministic=True)
            
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            # Ghi nhận Shanten sau mỗi lượt đánh
            shanten_history.append(info['shanten'])

        # 4. Trích xuất 5 bước cuối cùng của ván đấu
        # Nếu ván đấu diễn ra nhanh hơn 5 bước, nó sẽ lấy toàn bộ ván
        last_5_steps = shanten_history[-5:]
        
        # Cộng dồn vào bộ đếm tổng
        for s in last_5_steps:
            if s in total_shanten_counts:
                total_shanten_counts[s] += 1

    # 5. Tính toán trung bình và in kết quả
    print("\n" + "="*50)
    print(f" BÁO CÁO: TẦN SUẤT SHANTEN TRONG 5 BƯỚC CUỐI ({num_episodes} VÁN) ")
    print("="*50)
    print("Mỗi ván có tối đa 5 bước cuối được xét. Dưới đây là\ntrung bình số lượt/ván mà AI đạt mức Shanten tương ứng:\n")

    # In đúng các Shanten từ 1 đến 6 theo yêu cầu
    for shanten_level in range(1, 7):
        # Tính trung bình: Tổng số lượt xuất hiện chia cho số ván
        average_per_episode = total_shanten_counts[shanten_level] / num_episodes
        print(f" 🔹 Shanten {shanten_level}: {average_per_episode:>4.2f} lần / ván")
        
    print("-" * 50)
    # Khuyến mãi thêm thông số của Tenpai và Tới bài để bạn xem tỷ lệ thắng
    avg_tenpai = total_shanten_counts[0] / num_episodes
    avg_win = total_shanten_counts[-1] / num_episodes
    print(f" 🎯 Shanten  0 (Tenpai) : {avg_tenpai:>4.2f} lần / ván")
    print(f" 🏆 Shanten -1 (Tới bài): {avg_win:>4.2f} lần / ván")
    print("="*50)

if __name__ == "__main__":
    # ĐỪNG QUÊN SỬA LẠI ĐƯỜNG DẪN MODEL NẾU CẦN
    MODEL_FILE = "models/checkpoints_1/mahjong_model_1000000_steps.zip" 
    
    # Chạy thử 100 ván
    evaluate_last_5_steps(MODEL_FILE, num_episodes=100)